In [2]:
import sys, importlib
sys.path.insert(0, '..')

# Reload everything fresh
import src.auto_mapper, src.extractor, src.metric_builder
import src.calculation_engine, src.vector_pipeline, src.answer_engine
import pipeline

for mod in [src.auto_mapper, src.extractor, src.metric_builder,
            src.calculation_engine, src.vector_pipeline, src.answer_engine, pipeline]:
    importlib.reload(mod)

from pipeline import run_full_pipeline, ask
print("Ready.")


d:\sudhendra\learning projects\GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready.


In [3]:
# ============================================================
# CHANGE THESE 3 LINES FOR EACH NEW COMPANY
# ============================================================
COMPANY  = "tesla"
PDF_PATH = "../data/Tesla/tesla 22 10-k.pdf"
YEAR     = 2022
# ============================================================
print(f"Company: {COMPANY.upper()} | Year: {YEAR}")


Company: TESLA | Year: 2022


In [4]:
result = run_full_pipeline(
    company_name = COMPANY,
    pdf_path     = PDF_PATH,
    year         = YEAR,
)



  FULL PIPELINE: TESLA  |  FY2022

[Step 1/5] Detecting financial statement pages...
  Auto-detected income statement pages : [49, 80]
  Auto-detected balance sheet pages    : [48]
  Combined target pages: [48, 49, 80]

[Step 2/5] Loading/generating metric mapping config...
  Existing mapping config found: tesla.json

[Step 3/5] Extracting metrics from PDF...

  Company : TESLA
  PDF     : tesla 22 10-k.pdf
  Year    : 2022  |  Columns: [2022, 2021, 2020]
  Pages   : [48, 49, 80]

[Step 1/4] Extracting financial lines from PDF...
  Opened PDF: tesla 22 10-k.pdf (251 pages total)
  Extracted 66 candidate financial lines (74 rows skipped)

[Step 2/4] Assigning fiscal years to numeric columns...
  Year mapping complete: 66 rows with 3-year values

[Step 3/4] Saving raw extraction CSV for inspection...
  [Debug CSV saved] -> ..\data\Tesla\tesla_2022_raw_extraction.csv

[Step 4/4] Normalizing labels using mapping config...

Normalizing 66 rows for 'tesla'...
  [exact]  'cash and cash equiv

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7246.48it/s]


  Found existing collection 'tesla_2022' (2103 documents)
  Collection already populated. Skipping embedding step.
  (Pass rebuild=True to force re-embedding)

  Pipeline complete: TESLA FY2022
  Metrics ready    : ['cash_and_equivalents', 'total_assets', 'total_liabilities', 'total_revenue', 'gross_profit', 'research_and_development', 'selling_general_administrative', 'total_operating_expenses', 'operating_income', 'net_income']
  Vector store     : 2103 documents in ChromaDB



In [5]:
questions = [
    f"What was {COMPANY.title()}'s gross margin in {YEAR}?",
    f"What was the net profit margin in {YEAR}?",
    f"What was the revenue growth in {YEAR}?",
    f"What was the R&D as a percentage of revenue?",
]
print("DETERMINISTIC ANSWERS")
print("=" * 50)
for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q, result, use_llm=False)}")


DETERMINISTIC ANSWERS

Q: What was Tesla's gross margin in 2022?
A: Gross margin percentage in 2022 was 25.60%.

Q: What was the net profit margin in 2022?
A: Net profit margin in 2022 was 15.45%.

Q: What was the revenue growth in 2022?
A: Total net sales changed by 51.35% in 2022 compared with 2021.

Q: What was the R&D as a percentage of revenue?
A: R&D expense as a percentage of sales in 2022 was 3.77%.


In [6]:
rag_questions = [
    f"What are the main products of {COMPANY.title()}?",
    f"What risks does {COMPANY.title()} mention?",
    f"What is {COMPANY.title()}'s strategy for growth?",
    f"what is {COMPANY.title()}'s profit in year 2022 in percentages and amount too?",
]
print("RAG + LLM ANSWERS")
print("=" * 50)
for q in rag_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q, result)}")
    print("-" * 40)


RAG + LLM ANSWERS

Q: What are the main products of Tesla?
A: The main products of Tesla, according to the context provided, are high-performance fully electric vehicles and energy generation and storage systems.
----------------------------------------

Q: What risks does Tesla mention?
A: Tesla mentions several risks in the FY2022 10-K annual report. These include:

1. Operating in a cyclical industry that is sensitive to political and regulatory uncertainty, including with respect to trade and the environment, which can be compounded by inflationary pressures, rising energy prices, increases in interest rates, and any future global impact from the COVID-19 pandemic (Source 1).
2. Managing risks related to planned high-volume product sales, market and geographical expansion, and technological innovations. If they are not successful in managing these risks, their business, financial condition, and operating results may be harmed (Source 2).
3. Employee turnover due to a very competiti

In [7]:
print(ask("What did the company say about future growth?", result))


The report indicates that the company is focused on growing their manufacturing capacity, which includes ramping all production vehicles to their installed production capacities, increasing production rate, efficiency, and capacity at current factories. The next phase of production growth will depend on the ramp at Gigafactory Berlin-Brandenburg and Gigafactory Texas, as well as their ability to add to their available sources of battery cell supply by manufacturing their own cells. They also mention an emphasis on expanding operations to enable increased deliveries and deployments of their products for further revenue growth. However, the report cautions that quarter-to-quarter comparisons of financial results may not be consistent or linear due to factors such as introducing existing products to new markets, developing and introducing new products, and potential impacts from events like the COVID-19 pandemic.


In [9]:
import json
from pathlib import Path
from src.comparator import build_comparison_table, generate_narrative

# ── Load metrics from disk for each company ───────────────────────────────────
# Add or remove companies here. Each must have a saved metrics JSON.
COMPARE_YEAR = 2022

companies_to_compare = {
    "apple": json.loads(Path("../data/Apple/apple_2023_financial_metrics.json").read_text()),
    "tesla": json.loads(Path("../data/Tesla/tesla_2022_financial_metrics.json").read_text()),
}

print(f"Comparing {list(companies_to_compare.keys())} for year {COMPARE_YEAR}")


Comparing ['apple', 'tesla'] for year 2022


In [10]:
# Build comparison table
df = build_comparison_table(companies_to_compare, year=COMPARE_YEAR)
print("=" * 60)
print(f"FINANCIAL COMPARISON — FY{COMPARE_YEAR}")
print("=" * 60)
print(df.to_string())

# LLM narrative
print("\n" + "=" * 60)
print("ANALYST SUMMARY")
print("=" * 60)
narrative = generate_narrative(df, year=COMPARE_YEAR)
print(narrative)


FINANCIAL COMPARISON — FY2022
                                        apple         tesla
metric                                                     
total_revenue                   394328.000000  81462.000000
gross_profit                    170782.000000  20853.000000
operating_income                119437.000000  13656.000000
net_income                       99803.000000  12587.000000
research_and_development         26251.000000   3075.000000
selling_general_administrative   25094.000000   3946.000000
total_assets                    352755.000000  82338.000000
total_liabilities               302083.000000  26709.000000
gross_margin_%                      43.309631     25.598439
operating_margin_%                  30.288744     16.763644
net_margin_%                        25.309641     15.451376

ANALYST SUMMARY
 1. Apple had the highest revenue and profit in fiscal year 2022, with a total revenue of $394,328 million and net income of $99,803 million compared to Tesla's revenue of $

In [19]:
import importlib
import src.comparator
importlib.reload(src.comparator)

from src.comparator import compare_ask


In [20]:
apple_metrics = json.loads(
    Path("../data/Apple/apple_2023_financial_metrics.json").read_text()
)

tesla_metrics = json.loads(
    Path("../data/Tesla/tesla_2022_financial_metrics.json").read_text()
)

print("Apple keys:", list(apple_metrics.keys()))
print("Tesla keys:", list(tesla_metrics.keys()))


Apple keys: ['total_revenue', 'gross_profit', 'research_and_development', 'selling_general_administrative', 'total_operating_expenses', 'operating_income', 'net_income', 'cash_and_equivalents', 'total_assets', 'total_liabilities']
Tesla keys: ['cash_and_equivalents', 'total_assets', 'total_liabilities', 'total_revenue', 'gross_profit', 'research_and_development', 'selling_general_administrative', 'total_operating_expenses', 'operating_income', 'net_income']


In [22]:
from src.comparator import compare_ask

companies = {
    "apple": apple_metrics,
    "tesla": tesla_metrics,
}

print(compare_ask("Compare total assets between companies", companies, 2022))
print("---")
print(compare_ask("Which company is more efficient overall?", companies, 2022))


Total Assets (2022):
  Apple: 352755.00
  Tesla: 82338.00
---
 1. Apple had the highest revenue and profit in FY 2022, with total revenue of $394,328 million and net income of $99,803 million, compared to Tesla's total revenue of $81,462 million and net income of $12,587 million.

2. Apple was more efficient in FY 2022 as demonstrated by higher margins: gross margin of 43.3%, operating margin of 30.3%, and net margin of 25.3%. Tesla's corresponding figures were 25.6% for gross margin, 16.8% for operating margin, and 15.5% for net margin.

3. A notable difference between the companies is Apple's significantly higher total revenue and profit compared to Tesla, while also maintaining higher margins across all three categories (gross, operating, and net). This suggests that Apple has a more established business model with greater financial stability, as compared to Tesla which is still focused on growth and innovation in the automotive sector.
